# 5. Historical Word Embedding Models

Train Word2Vec models on period-specific corpora (EEBO-TCP, ECCO-TCP, COHA)
and compute how abstract/concrete word meanings shift across centuries.

In [ ]:
from abstraction.models import (
    get_model_paths, load_model, gen_skipgrams_corpus,
    gen_model, gen_vecnorms,
)
from abstraction.norms import get_vecnorms, get_allnorms, corr_norms
import pandas as pd

## Inspect available models

In [ ]:
paths = get_model_paths()
if paths:
    df_paths = pd.DataFrame(paths)
    print(f'{len(df_paths)} model files found')
    df_paths.groupby(['corpus', 'period_start', 'period_end']).size().reset_index(name='runs')
else:
    print('No models found in data/models/. Generate skipgrams first.')

## Load a model and explore word neighborhoods

In [ ]:
if paths:
    model = load_model(paths[0]['path'])
    kv = model.wv if hasattr(model, 'wv') else model
    print(f'Vocabulary size: {len(kv)}')
    
    for word in ['virtue', 'justice', 'rock', 'face']:
        if word in kv:
            neighbors = kv.most_similar(word, topn=5)
            print(f'\n{word}: {[(w, round(s, 3)) for w, s in neighbors]}')

## Vector-based norms

These are concreteness scores derived from historical models rather than modern surveys.
They show how abstract/concrete a word was *in its historical usage context*.

In [ ]:
try:
    vecnorms = get_vecnorms()
    print(f'{len(vecnorms)} words, {len(vecnorms.columns)} period-source columns')
    vecnorms.head()
except FileNotFoundError:
    print('Vector norms not yet generated. Run gen_vecnorms() first.')

## Correlation: empirical vs. historical norms

In [ ]:
try:
    allnorms = get_allnorms()
    corr_df = corr_norms(allnorms)
    print(f'Min r = {corr_df["value"].min():.3f}, Median r = {corr_df["value"].median():.3f}, Max r = {corr_df["value"].max():.3f}')
    corr_df.head(10)
except FileNotFoundError:
    print('All-norms file not found. Generate empirical and vector norms first.')

## Pipeline: generating from scratch

The full pipeline for creating models and vector norms. Each step is expensive and
writes results to disk, so they only need to run once.

In [ ]:
# Step 1: Generate skipgrams from corpus text files
# gen_skipgrams_corpus('eebo_tcp', min_year=1500, max_year=1700, num_proc=4)
# gen_skipgrams_corpus('ecco_tcp', min_year=1700, max_year=1800, num_proc=4)
# gen_skipgrams_corpus('coha', min_year=1800, max_year=2000, num_proc=4)

In [ ]:
# Step 2: Train Word2Vec models (multiple runs per period)
# from abstraction.models import gen_models_corpus
# gen_models_corpus('eebo_tcp', num_runs=10, num_proc=4)
# gen_models_corpus('ecco_tcp', num_runs=10, num_proc=4)
# gen_models_corpus('coha', num_runs=10, num_proc=4)

In [ ]:
# Step 3: Compute vector-based norms from trained models
# gen_vecnorms()